In [1]:
!apt-get update -qq
!apt-get install -y -qq openmpi-bin libopenmpi-dev
!pip install -q mpi4py numba numpy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!nvidia-smi
!mpirun --version
!python -c "import mpi4py, numba, numpy; print('All Python packages installed successfully')"

Fri Nov 20 03:42:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda
import numba
import numpy as np
import subprocess

GITHUB_USERNAME = "230103369-bit"

print("GitHub:", GITHUB_USERNAME)
print("Numba version:", numba.__version__)
print("NumPy version:", np.__version__)
print("CUDA available:", cuda.is_available())

if cuda.is_available():
    gpu = cuda.get_current_device()
    print("GPU name:", gpu.name)
    print("Compute capability:", gpu.compute_capability)

print("GPU UUID:")
subprocess.run([
    "nvidia-smi",
    "--query-gpu=uuid",
    "--format=csv,noheader"
])

GitHub: 230103369-bit
Numba version: 0.64.0
NumPy version: 2.4.6
CUDA available: True
GPU name: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
GPU UUID:


CompletedProcess(args=['nvidia-smi', '--query-gpu=uuid', '--format=csv,noheader'], returncode=0)

In [4]:
%%writefile mpi_test.py
from mpi4py import MPI
from numba import cuda

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

cuda.select_device(0)

print(
    f"Rank {rank}/{size} | "
    f"GPU: {cuda.get_current_device().name}",
    flush=True
)

comm.Barrier()

if rank == 0:
    print("MPI + CUDA TEST PASSED", flush=True)

Writing mpi_test.py


In [5]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python mpi_test.py

Rank 2/4 | GPU: Tesla T4
Rank 3/4 | GPU: Tesla T4
Rank 1/4 | GPU: Tesla T4
Rank 0/4 | GPU: Tesla T4
MPI + CUDA TEST PASSED


In [6]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import sys
import hashlib
import platform
import subprocess

import numpy as np
from mpi4py import MPI
from numba import cuda

GITHUB_USERNAME = "230103369-bit"

def init_hybrid_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError("Exactly 4 MPI ranks are required")

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False
    )

    up, down = cart_comm.Shift(0, 1)
    left, right = cart_comm.Shift(1, 1)

    cuda.select_device(0)

    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    node_size = node_comm.Get_size()

    if node_size != 4:
        raise RuntimeError("All 4 ranks must share one node")

    # Rank 0 owns the shared allocation.
    # Extra space permits explicit 64-byte alignment.
    nbytes = 64 * np.dtype(np.uint64).itemsize + 63
    win = MPI.Win.Allocate_shared(
        nbytes if node_rank == 0 else 0,
        1,
        comm=node_comm
    )

    raw, disp_unit = win.Shared_query(0)
    base = np.frombuffer(raw, dtype=np.uint8)

    address = base.ctypes.data
    offset = (-address) % 64

    shm_view = np.frombuffer(
        raw,
        dtype=np.uint64,
        count=64,
        offset=offset
    )

    assert shm_view.ctypes.data % 64 == 0

    gpu_uuid = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        text=True
    ).strip().splitlines()[0]

    cpu_signature = f"{platform.machine()}::{os.cpu_count()}"

    seed_material = (
        f"{github_user}::{cpu_signature}::{gpu_uuid}"
    )

    instance_seed = int(
        hashlib.sha256(seed_material.encode()).hexdigest()[:16],
        16
    )

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "node_comm": node_comm,
        "win": win,
        "shm_view": shm_view,
        "rank": rank,
        "size": size,
        "coords": cart_comm.Get_coords(rank),
        "neighbors": {
            "up": up,
            "down": down,
            "left": left,
            "right": right
        },
        "gpu_uuid": gpu_uuid,
        "cpu_signature": cpu_signature,
        "instance_seed": instance_seed
    }


if __name__ == "__main__":
    ctx = init_hybrid_context(GITHUB_USERNAME)

    for r in range(4):
        ctx["comm"].Barrier()

        if ctx["rank"] == r:
            print(
                f"Rank {r} | "
                f"Coords {ctx['coords']} | "
                f"Neighbors {ctx['neighbors']} | "
                f"Shared aligned: "
                f"{ctx['shm_view'].ctypes.data % 64 == 0}",
                flush=True
            )

    ctx["comm"].Barrier()

    if ctx["rank"] == 0:
        print("TASK 1 PASSED", flush=True)

    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing hybrid_mpi_cuda_cache.py


In [7]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python hybrid_mpi_cuda_cache.py

Rank 0 | Coords [0, 0] | Neighbors {'up': -2, 'down': 2, 'left': -2, 'right': 1} | Shared aligned: True
Rank 1 | Coords [0, 1] | Neighbors {'up': -2, 'down': 3, 'left': 0, 'right': -2} | Shared aligned: True
Rank 2 | Coords [1, 0] | Neighbors {'up': 0, 'down': -2, 'left': -2, 'right': 3} | Shared aligned: True
Rank 3 | Coords [1, 1] | Neighbors {'up': 1, 'down': -2, 'left': 2, 'right': -2} | Shared aligned: True
TASK 1 PASSED


In [8]:
%%writefile -a hybrid_mpi_cuda_cache.py


# ============================================================
# TASK 2 — GPU FIELD INITIALIZATION AND HALO PACK/UNPACK
# ============================================================

from numba import float64

LOCAL_N = 256
PADDED_N = LOCAL_N + 2


def initialize_fields(coords):
    row_rank, col_rank = coords

    u = np.zeros((PADDED_N, PADDED_N), dtype=np.float64)
    kappa = np.zeros_like(u)
    source = np.zeros_like(u)

    dx = 1.0 / 513.0

    for i in range(1, LOCAL_N + 1):
        for j in range(1, LOCAL_N + 1):
            global_i = row_rank * LOCAL_N + i
            global_j = col_rank * LOCAL_N + j

            x = global_j * dx
            y = global_i * dx

            u[i, j] = np.sin(np.pi * x) * np.sin(np.pi * y)
            kappa[i, j] = 0.8 + 0.2 * np.sin(np.pi * x) ** 2
            source[i, j] = 0.01 * np.cos(np.pi * x) * np.cos(np.pi * y)

    return u, kappa, source


@cuda.jit
def pack_halos_kernel(field, top, bottom, left, right):
    idx = cuda.grid(1)

    if idx < LOCAL_N:
        j = idx + 1

        top[idx] = field[1, j]
        bottom[idx] = field[LOCAL_N, j]

        left[idx] = field[j, 1]
        right[idx] = field[j, LOCAL_N]


@cuda.jit
def unpack_halos_kernel(field, top, bottom, left, right):
    idx = cuda.grid(1)

    if idx < LOCAL_N:
        j = idx + 1

        field[0, j] = top[idx]
        field[LOCAL_N + 1, j] = bottom[idx]

        field[j, 0] = left[idx]
        field[j, LOCAL_N + 1] = right[idx]


def test_task2(ctx):
    u, kappa, source = initialize_fields(ctx["coords"])

    d_u = cuda.to_device(u)

    send = [
        cuda.device_array(LOCAL_N, dtype=np.float64)
        for _ in range(4)
    ]

    recv = [
        cuda.to_device(np.full(LOCAL_N, float(i + 1)))
        for i in range(4)
    ]

    threads = 128
    blocks = (LOCAL_N + threads - 1) // threads

    pack_halos_kernel[blocks, threads](d_u, *send)
    cuda.synchronize()

    packed = [buf.copy_to_host() for buf in send]

    assert np.allclose(packed[0], u[1, 1:257])
    assert np.allclose(packed[1], u[256, 1:257])
    assert np.allclose(packed[2], u[1:257, 1])
    assert np.allclose(packed[3], u[1:257, 256])

    unpack_halos_kernel[blocks, threads](d_u, *recv)
    cuda.synchronize()

    result = d_u.copy_to_host()

    assert np.all(result[0, 1:257] == 1.0)
    assert np.all(result[257, 1:257] == 2.0)
    assert np.all(result[1:257, 0] == 3.0)
    assert np.all(result[1:257, 257] == 4.0)

    return {
        "rank": ctx["rank"],
        "field_shape": list(u.shape),
        "kappa_min": float(kappa[1:257, 1:257].min()),
        "kappa_max": float(kappa[1:257, 1:257].max()),
        "pack_passed": True,
        "unpack_passed": True,
    }

Appending to hybrid_mpi_cuda_cache.py


In [9]:
%%writefile task2_test.py
from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    test_task2,
)

ctx = init_hybrid_context("230103369-bit")

result = test_task2(ctx)

for rank in range(4):
    ctx["comm"].Barrier()
    if ctx["rank"] == rank:
        print(result, flush=True)

ctx["comm"].Barrier()

if ctx["rank"] == 0:
    print("TASK 2 PASSED", flush=True)

ctx["win"].Free()
ctx["node_comm"].Free()
ctx["cart_comm"].Free()

Writing task2_test.py


In [10]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task2_test.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

In [11]:
%%writefile -a hybrid_mpi_cuda_cache.py


# ============================================================
# TASK 3 — 16x16 SHARED-MEMORY TILED STENCIL
# ============================================================

TILE_DIM = 16
SHARED_DIM = 18
LAMBDA_DAMP = 0.15


@cuda.jit
def tiled_stencil_kernel(u, kappa, source, out, dt, dx,
                         row_begin, row_end, col_begin, col_end):
    tile = cuda.shared.array((18, 18), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + 1
    col = cuda.blockIdx.x * 16 + tx + 1

    sy = ty + 1
    sx = tx + 1

    # Center
    tile[sy, sx] = u[row, col]

    # Four halo sides
    if tx == 0:
        tile[sy, 0] = u[row, col - 1]

    if tx == 15:
        tile[sy, 17] = u[row, col + 1]

    if ty == 0:
        tile[0, sx] = u[row - 1, col]

    if ty == 15:
        tile[17, sx] = u[row + 1, col]

    cuda.syncthreads()

    if (row_begin <= row <= row_end and
        col_begin <= col <= col_end and
        row <= 256 and col <= 256):

        center = tile[sy, sx]

        laplacian = (
            tile[sy - 1, sx] +
            tile[sy + 1, sx] +
            tile[sy, sx - 1] +
            tile[sy, sx + 1] -
            4.0 * center
        ) / (dx * dx)

        out[row, col] = (
            center +
            dt * (
                kappa[row, col] * laplacian +
                source[row, col] -
                LAMBDA_DAMP * center
            )
        )


def test_tiled_stencil(ctx):
    u, kappa, source = initialize_fields(ctx["coords"])

    d_u = cuda.to_device(u)
    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)
    d_out = cuda.to_device(u.copy())

    dx = 1.0 / 513.0
    kappa_max = float(kappa.max())
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = 0.5 * dt_max

    blocks = (16, 16)
    threads = (16, 16)

    tiled_stencil_kernel[blocks, threads](
        d_u, d_kappa, d_source, d_out,
        dt, dx, 1, 256, 1, 256
    )

    cuda.synchronize()

    result = d_out.copy_to_host()

    assert np.isfinite(result).all()
    assert not np.array_equal(result[1:257, 1:257],
                              u[1:257, 1:257])

    # Reference check for an interior point.
    i, j = 128, 128

    lap = (
        u[i - 1, j] + u[i + 1, j] +
        u[i, j - 1] + u[i, j + 1] -
        4.0 * u[i, j]
    ) / (dx * dx)

    expected = u[i, j] + dt * (
        kappa[i, j] * lap +
        source[i, j] -
        LAMBDA_DAMP * u[i, j]
    )

    error = abs(result[i, j] - expected)
    assert error < 1e-12, f"Stencil mismatch: {error}"

    ptx = tiled_stencil_kernel.inspect_asm()
    asm = "\n".join(ptx.values()) if isinstance(ptx, dict) else ptx

    import re

    audit = {
        "st_shared_f64": len(re.findall(r"\bst\.shared\.f64\b", asm)),
        "ld_shared_f64": len(re.findall(r"\bld\.shared\.f64\b", asm)),
        "bar_sync": len(re.findall(r"\bbar\.sync\b", asm)),
    }

    print(
        f"Rank {ctx['rank']} | "
        f"Stencil error: {error:.3e} | "
        f"PTX audit: {audit}",
        flush=True
    )

    assert audit["bar_sync"] > 0, "Missing CUDA synchronization"
    assert audit["st_shared_f64"] > 0, "Missing shared-memory stores"
    assert audit["ld_shared_f64"] > 0, "Missing shared-memory loads"

    return audit

Appending to hybrid_mpi_cuda_cache.py


In [12]:
%%writefile task3_test.py
from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    test_tiled_stencil,
)

ctx = init_hybrid_context("230103369-bit")

audit = test_tiled_stencil(ctx)

ctx["comm"].Barrier()

if ctx["rank"] == 0:
    print("TASK 3 — TILED STENCIL TEST PASSED", flush=True)

ctx["win"].Free()
ctx["node_comm"].Free()
ctx["cart_comm"].Free()

Writing task3_test.py


In [13]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task3_test.py

Rank 0 | Stencil error: 0.000e+00 | PTX audit: {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
Rank 3 | Stencil error: 0.000e+00 | PTX audit: {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
Rank 2 | Stencil error: 0.000e+00 | PTX audit: {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
Rank 1 | Stencil error: 0.000e+00 | PTX audit: {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
TASK 3 — TILED STENCIL TEST PASSED


In [14]:
%%writefile -a hybrid_mpi_cuda_cache.py


# ============================================================
# TASK 3B — MPI HALO EXCHANGE + DUAL CUDA STREAMS
# ============================================================

import time

DIRECTIONS = ("up", "down", "left", "right")

SEND_TAGS = {
    "up": 10,
    "down": 11,
    "left": 12,
    "right": 13,
}

OPPOSITE = {
    "up": "down",
    "down": "up",
    "left": "right",
    "right": "left",
}


def create_pipeline(ctx):
    u, kappa, source = initialize_fields(ctx["coords"])

    dx = 1.0 / 513.0

    local_max = float(kappa[1:257, 1:257].max())
    kappa_max = ctx["cart_comm"].allreduce(
        local_max, op=MPI.MAX
    )

    dt_max = dx * dx / (4.0 * kappa_max)
    dt = 0.5 * dt_max

    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    return {
        "u": cuda.to_device(u),
        "out": cuda.to_device(u.copy()),
        "kappa": cuda.to_device(kappa),
        "source": cuda.to_device(source),
        "dx": dx,
        "dt": dt,
        "dt_max": dt_max,
        "kappa_max": kappa_max,
        "stream_interior": stream_interior,
        "stream_boundary": stream_boundary,
        "send_gpu": {
            direction: cuda.device_array(
                256, dtype=np.float64
            )
            for direction in DIRECTIONS
        },
        "recv_gpu": {
            direction: cuda.device_array(
                256, dtype=np.float64
            )
            for direction in DIRECTIONS
        },
        "send_host": {
            direction: cuda.pinned_array(
                256, dtype=np.float64
            )
            for direction in DIRECTIONS
        },
        "recv_host": {
            direction: cuda.pinned_array(
                256, dtype=np.float64
            )
            for direction in DIRECTIONS
        },
    }


def launch_stencil(pipeline, stream, region):
    if region == "interior":
        bounds = (2, 255, 2, 255)
    elif region == "border":
        bounds = (1, 256, 1, 256)
    else:
        raise ValueError(region)

    if region == "interior":
        tiled_stencil_kernel[(16, 16), (16, 16), stream](
            pipeline["u"],
            pipeline["kappa"],
            pipeline["source"],
            pipeline["out"],
            pipeline["dt"],
            pipeline["dx"],
            *bounds
        )
    else:
        border_stencil_kernel[(16, 16), (16, 16), stream](
            pipeline["u"],
            pipeline["kappa"],
            pipeline["source"],
            pipeline["out"],
            pipeline["dt"],
            pipeline["dx"]
        )


@cuda.jit
def border_stencil_kernel(u, kappa, source, out, dt, dx):
    tile = cuda.shared.array((18, 18), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + 1
    col = cuda.blockIdx.x * 16 + tx + 1

    sy = ty + 1
    sx = tx + 1

    tile[sy, sx] = u[row, col]

    if tx == 0:
        tile[sy, 0] = u[row, col - 1]
    if tx == 15:
        tile[sy, 17] = u[row, col + 1]
    if ty == 0:
        tile[0, sx] = u[row - 1, col]
    if ty == 15:
        tile[17, sx] = u[row + 1, col]

    cuda.syncthreads()

    if row == 1 or row == 256 or col == 1 or col == 256:
        center = tile[sy, sx]

        lap = (
            tile[sy - 1, sx] +
            tile[sy + 1, sx] +
            tile[sy, sx - 1] +
            tile[sy, sx + 1] -
            4.0 * center
        ) / (dx * dx)

        out[row, col] = center + dt * (
            kappa[row, col] * lap +
            source[row, col] -
            LAMBDA_DAMP * center
        )


def exchange_halos(ctx, pipeline, overlapped):
    cart = ctx["cart_comm"]
    neighbors = ctx["neighbors"]

    boundary_stream = pipeline["stream_boundary"]
    interior_stream = pipeline["stream_interior"]

    requests = []

    # Post all receives first.
    for direction in DIRECTIONS:
        neighbor = neighbors[direction]

        if neighbor != MPI.PROC_NULL:
            req = cart.Irecv(
                pipeline["recv_host"][direction],
                source=neighbor,
                tag=SEND_TAGS[OPPOSITE[direction]]
            )
            requests.append(req)
        else:
            pipeline["recv_host"][direction].fill(0.0)

    # Pack outgoing halos on the GPU.
    pack_halos_kernel[2, 128, boundary_stream](
        pipeline["u"],
        pipeline["send_gpu"]["up"],
        pipeline["send_gpu"]["down"],
        pipeline["send_gpu"]["left"],
        pipeline["send_gpu"]["right"]
    )

    for direction in DIRECTIONS:
        if neighbors[direction] != MPI.PROC_NULL:
            pipeline["send_gpu"][direction].copy_to_host(
                pipeline["send_host"][direction],
                stream=boundary_stream
            )

    # MPI cannot send a buffer before its D2H transfer finishes.
    boundary_stream.synchronize()

    for direction in DIRECTIONS:
        neighbor = neighbors[direction]

        if neighbor != MPI.PROC_NULL:
            req = cart.Isend(
                pipeline["send_host"][direction],
                dest=neighbor,
                tag=SEND_TAGS[direction]
            )
            requests.append(req)

    # The deep interior does not depend on incoming halos.
    if overlapped:
        launch_stencil(pipeline, interior_stream, "interior")

    MPI.Request.Waitall(requests)

    # Copy received host buffers to GPU.
    for direction in DIRECTIONS:
        pipeline["recv_gpu"][direction].copy_to_device(
            pipeline["recv_host"][direction],
            stream=boundary_stream
        )

    unpack_halos_kernel[2, 128, boundary_stream](
        pipeline["u"],
        pipeline["recv_gpu"]["up"],
        pipeline["recv_gpu"]["down"],
        pipeline["recv_gpu"]["left"],
        pipeline["recv_gpu"]["right"]
    )

    if not overlapped:
        boundary_stream.synchronize()
        launch_stencil(pipeline, interior_stream, "interior")
        interior_stream.synchronize()

    # Border calculation waits for incoming halo unpacking.
    launch_stencil(pipeline, boundary_stream, "border")

    # Both streams must finish before swapping fields.
    interior_stream.synchronize()
    boundary_stream.synchronize()

    pipeline["u"], pipeline["out"] = (
        pipeline["out"], pipeline["u"]
    )


def step_serialized(ctx, pipeline):
    exchange_halos(ctx, pipeline, overlapped=False)


def step_overlapped(ctx, pipeline):
    exchange_halos(ctx, pipeline, overlapped=True)


def test_dual_stream(ctx):
    serial = create_pipeline(ctx)
    overlap = create_pipeline(ctx)

    cart = ctx["cart_comm"]

    cart.Barrier()
    start = time.perf_counter()

    for _ in range(3):
        step_serialized(ctx, serial)

    cart.Barrier()
    serial_ms = (time.perf_counter() - start) * 1000

    cart.Barrier()
    start = time.perf_counter()

    for _ in range(3):
        step_overlapped(ctx, overlap)

    cart.Barrier()
    overlap_ms = (time.perf_counter() - start) * 1000

    a = serial["u"].copy_to_host()
    b = overlap["u"].copy_to_host()

    local_diff = float(
        np.max(np.abs(a[1:257, 1:257] - b[1:257, 1:257]))
    )

    max_diff = cart.allreduce(local_diff, op=MPI.MAX)

    assert max_diff < 1e-12, (
        f"Serialized/overlapped mismatch: {max_diff}"
    )

    max_serial_ms = cart.allreduce(serial_ms, op=MPI.MAX)
    max_overlap_ms = cart.allreduce(overlap_ms, op=MPI.MAX)

    if ctx["rank"] == 0:
        print("DUAL STREAM TEST RESULTS")
        print(f"Serialized: {max_serial_ms:.3f} ms")
        print(f"Overlapped: {max_overlap_ms:.3f} ms")
        print(f"Maximum difference: {max_diff:.3e}")
        print("TASK 3B PASSED", flush=True)

    return {
        "serialized_ms": max_serial_ms,
        "overlapped_ms": max_overlap_ms,
        "max_diff": max_diff
    }

Appending to hybrid_mpi_cuda_cache.py


In [15]:
%%writefile task3b_test.py

from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    test_dual_stream
)

ctx = init_hybrid_context("230103369-bit")

try:
    result = test_dual_stream(ctx)
finally:
    ctx["comm"].Barrier()
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing task3b_test.py


In [16]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task3b_test.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

In [17]:
%%writefile -a hybrid_mpi_cuda_cache.py


# ============================================================
# TASK 3C — WARMED 300-STEP BENCHMARK
# ============================================================

def benchmark_300_steps(ctx):
    cart = ctx["cart_comm"]
    rank = ctx["rank"]

    serial = create_pipeline(ctx)
    overlap = create_pipeline(ctx)

    # Compile all CUDA kernels before benchmarking.
    warmup_serial = create_pipeline(ctx)
    warmup_overlap = create_pipeline(ctx)

    step_serialized(ctx, warmup_serial)
    step_overlapped(ctx, warmup_overlap)

    cuda.synchronize()
    cart.Barrier()

    def run_mode(pipeline, step_function):
        cart.Barrier()
        start = time.perf_counter()

        for _ in range(300):
            step_function(ctx, pipeline)

        cuda.synchronize()
        cart.Barrier()

        local_ms = (time.perf_counter() - start) * 1000

        return cart.allreduce(local_ms, op=MPI.MAX)

    serialized_ms = run_mode(serial, step_serialized)
    overlapped_ms = run_mode(overlap, step_overlapped)

    a = serial["u"].copy_to_host()
    b = overlap["u"].copy_to_host()

    local_error = float(
        np.max(
            np.abs(
                a[1:257, 1:257] -
                b[1:257, 1:257]
            )
        )
    )

    max_error = cart.allreduce(local_error, op=MPI.MAX)

    local_energy_serial = float(
        np.sum(a[1:257, 1:257] ** 2)
    )

    local_energy_overlap = float(
        np.sum(b[1:257, 1:257] ** 2)
    )

    energy_serial = cart.allreduce(
        local_energy_serial, op=MPI.SUM
    )

    energy_overlap = cart.allreduce(
        local_energy_overlap, op=MPI.SUM
    )

    assert max_error < 1e-12, (
        f"Parity check failed: {max_error}"
    )

    speedup = serialized_ms / overlapped_ms

    if rank == 0:
        print("\nTASK 3C — 300-STEP BENCHMARK")
        print(f"Serialized: {serialized_ms:.3f} ms")
        print(f"Overlapped: {overlapped_ms:.3f} ms")
        print(f"Speedup: {speedup:.4f}x")
        print(f"Maximum error: {max_error:.3e}")
        print(f"Serialized energy: {energy_serial:.12f}")
        print(f"Overlapped energy: {energy_overlap:.12f}")
        print("TASK 3C PASSED", flush=True)

    return {
        "steps": 300,
        "serialized_ms": serialized_ms,
        "overlapped_ms": overlapped_ms,
        "speedup": speedup,
        "max_error": max_error,
        "serialized_energy": energy_serial,
        "overlapped_energy": energy_overlap,
        "verification_passed": max_error < 1e-12,
    }

Appending to hybrid_mpi_cuda_cache.py


In [18]:
%%writefile task3c_test.py
from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    benchmark_300_steps,
)

ctx = init_hybrid_context("230103369-bit")

try:
    benchmark_300_steps(ctx)
finally:
    ctx["comm"].Barrier()
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing task3c_test.py


In [19]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task3c_test.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

In [24]:
%%writefile -a hybrid_mpi_cuda_cache.py


# ============================================================
# TASK 4 — MPI FALSE SHARING BENCHMARK
# ============================================================

from numba import njit, types, intrinsic
from llvmlite import ir
import time

CACHE_ITERATIONS = 40_000_000
CACHE_STRIDES = (1, 8, 16)


@intrinsic
def mpi_atomicity_free_increment(typingctx, array, index):
    sig = types.void(array, index)

    def codegen(context, builder, signature, args):
        array_value, index_value = args

        ary = context.make_array(signature.args[0])(
            context, builder, array_value
        )

        idx = context.cast(
            builder,
            index_value,
            signature.args[1],
            types.intp
        )

        pointer = builder.gep(ary.data, [idx])

        barrier = ir.InlineAsm(
            ir.FunctionType(ir.VoidType(), []),
            "",
            "~{memory}",
            side_effect=True
        )

        builder.call(barrier, [])

        current = builder.load(pointer)
        current.volatile = True

        updated = builder.add(
            current,
            ir.Constant(current.type, 1)
        )

        store = builder.store(updated, pointer)
        store.volatile = True

        builder.call(barrier, [])

        return context.get_dummy_value()

    return sig, codegen


@njit(nogil=True)
def mpi_false_sharing_worker(shared, slot, iterations):
    for _ in range(iterations):
        mpi_atomicity_free_increment(shared, slot)


def benchmark_mpi_false_sharing(ctx):
    comm = ctx["comm"]
    node_comm = ctx["node_comm"]
    win = ctx["win"]
    shared = ctx["shm_view"]
    rank = ctx["rank"]

    # JIT compilation before measurement.
    dummy = np.zeros(64, dtype=np.uint64)
    mpi_false_sharing_worker(dummy, 0, 1)

    results = {}

    for stride in CACHE_STRIDES:
        # Reset the shared counters.
        if rank == 0:
            shared[:] = 0

        win.Sync()
        node_comm.Barrier()
        win.Sync()

        slot = rank * stride

        comm.Barrier()
        start = time.perf_counter_ns()

        mpi_false_sharing_worker(
            shared, slot, CACHE_ITERATIONS
        )

        win.Sync()
        comm.Barrier()

        local_ns = time.perf_counter_ns() - start
        elapsed_ns = comm.allreduce(local_ns, op=MPI.MAX)

        win.Sync()
        node_comm.Barrier()
        win.Sync()

        if rank == 0:
            counts = [
                int(shared[r * stride])
                for r in range(4)
            ]

            passed = all(
                count == CACHE_ITERATIONS
                for count in counts
            )

            results[str(stride)] = {
                "stride": stride,
                "elapsed_ns": int(elapsed_ns),
                "elapsed_ms": elapsed_ns / 1e6,
                "counts": counts,
                "verification_passed": passed
            }

            print(
                f"Stride {stride:2d} | "
                f"{elapsed_ns / 1e6:.3f} ms | "
                f"Counts: {counts} | "
                f"PASS: {passed}",
                flush=True
            )

            if not passed:
                raise RuntimeError(
                    f"Counter verification failed for stride {stride}"
                )

        comm.Barrier()

    if rank == 0:
        t1 = results["1"]["elapsed_ns"]
        t8 = results["8"]["elapsed_ns"]
        t16 = results["16"]["elapsed_ns"]

        print("\nTASK 4 — SUMMARY")
        print(f"Stride 1 / 8 slowdown:  {t1 / t8:.4f}x")
        print(f"Stride 1 / 16 slowdown: {t1 / t16:.4f}x")
        print("TASK 4 PASSED", flush=True)

    return results if rank == 0 else None

Appending to hybrid_mpi_cuda_cache.py


In [25]:
%%writefile task4_test.py

from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    benchmark_mpi_false_sharing
)

ctx = init_hybrid_context("230103369-bit")

try:
    results = benchmark_mpi_false_sharing(ctx)
finally:
    ctx["comm"].Barrier()
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Overwriting task4_test.py


In [26]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task4_test.py

Traceback (most recent call last):
  File "/content/task4_test.py", line 2, in <module>
    from hybrid_mpi_cuda_cache import (
    ...<2 lines>...
    )
  File "/content/hybrid_mpi_cuda_cache.py", line 776, in <module>
    from numba import njit, types, intrinsic
ImportError: cannot import name 'intrinsic' from 'numba' (/usr/local/lib/python3.13/dist-packages/numba/__init__.py)
Traceback (most recent call last):
  File "/content/task4_test.py", line 2, in <module>
    from hybrid_mpi_cuda_cache import (
    ...<2 lines>...
    )
  File "/content/hybrid_mpi_cuda_cache.py", line 776, in <module>
    from numba import njit, types, intrinsic
ImportError: cannot import name 'intrinsic' from 'numba' (/usr/local/lib/python3.13/dist-packages/numba/__init__.py)
Traceback (most recent call last):
  File "/content/task4_test.py", line 2, in <module>
    from hybrid_mpi_cuda_cache import (
    ...<2 lines>...
    )
  File "/content/hybrid_mpi_cuda_cache.py", line 776, in <module>
    from numba i

In [27]:
from pathlib import Path

path = Path("hybrid_mpi_cuda_cache.py")
code = path.read_text()

old = "from numba import njit, types, intrinsic"
new = "from numba import njit, types\nfrom numba.extending import intrinsic"

assert old in code, "Строка импорта не найдена"
path.write_text(code.replace(old, new))

print("Import fixed successfully")

Import fixed successfully


In [28]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task4_test.py

Stride  1 | 508.585 ms | Counts: [40000000, 40000000, 40000000, 40000000] | PASS: True
Stride  8 | 149.820 ms | Counts: [40000000, 40000000, 40000000, 40000000] | PASS: True
Stride 16 | 200.115 ms | Counts: [40000000, 40000000, 40000000, 40000000] | PASS: True

TASK 4 — SUMMARY
Stride 1 / 8 slowdown:  3.3946x
Stride 1 / 16 slowdown: 2.5415x
TASK 4 PASSED


In [29]:
import subprocess

print("CPU cache line size:")
subprocess.run(
    ["cat", "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"]
)

print("\nCPU model:")
subprocess.run(
    ["bash", "-lc", "lscpu | grep 'Model name'"]
)

CPU cache line size:

CPU model:


CompletedProcess(args=['bash', '-lc', "lscpu | grep 'Model name'"], returncode=0)

In [30]:
import subprocess

def shell_output(command):
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout.strip() or result.stderr.strip()

print("CPU MODEL:")
print(shell_output("lscpu | grep -E 'Model name|Architecture'"))

print("\nCACHE LINE SIZE:")
print(shell_output(
    "cat /sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
))

CPU MODEL:
Architecture:                            x86_64
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz

CACHE LINE SIZE:
64


In [31]:
import numpy as np
from hybrid_mpi_cuda_cache import mpi_false_sharing_worker

dummy = np.zeros(64, dtype=np.uint64)
mpi_false_sharing_worker(dummy, 0, 10)

signature = mpi_false_sharing_worker.signatures[0]
llvm = mpi_false_sharing_worker.inspect_llvm(signature)

import re

loads = re.findall(r'\bload i64\b', llvm)
stores = re.findall(r'\bstore i64\b', llvm)
barriers = re.findall(r'asm sideeffect', llvm)

print("LLVM AUDIT")
print("i64 loads:", len(loads))
print("i64 stores:", len(stores))
print("Memory barriers:", len(barriers))

assert len(loads) > 0
assert len(stores) > 0
assert len(barriers) > 0

print("LLVM AUDIT PASSED")

LLVM AUDIT
i64 loads: 2
i64 stores: 1
Memory barriers: 2
LLVM AUDIT PASSED


In [32]:
%%writefile task5_final.py

import json
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
from mpi4py import MPI
from numba import cuda

from hybrid_mpi_cuda_cache import (
    init_hybrid_context,
    benchmark_300_steps,
    mpi_false_sharing_worker,
    CACHE_ITERATIONS,
    CACHE_STRIDES,
)

USERNAME = "230103369-bit"
comm = MPI.COMM_WORLD
rank = comm.Get_rank()


def get_cache_line_size():
    try:
        path = (
            "/sys/devices/system/cpu/cpu0/cache/"
            "index0/coherency_line_size"
        )
        with open(path) as f:
            return int(f.read().strip())
    except (OSError, ValueError):
        return None


def audit_llvm():
    dummy = np.zeros(64, dtype=np.uint64)
    mpi_false_sharing_worker(dummy, 0, 10)

    signature = mpi_false_sharing_worker.signatures[0]
    llvm = mpi_false_sharing_worker.inspect_llvm(signature)

    import re

    loads = len(re.findall(r"\bload i64\b", llvm))
    stores = len(re.findall(r"\bstore i64\b", llvm))
    barriers = len(re.findall(r"asm sideeffect", llvm))

    passed = loads > 0 and stores > 0 and barriers > 0

    return {
        "i64_loads": loads,
        "i64_stores": stores,
        "memory_barriers": barriers,
        "basic_audit_passed": passed,
        "loop_store_placement_verified": False,
    }


ctx = init_hybrid_context(USERNAME)

try:
    # Re-run the complete warmed 300-step comparison.
    task3 = benchmark_300_steps(ctx)

    comm.Barrier()

    # LLVM inspection on each rank.
    llvm_audit = audit_llvm()
    all_llvm_pass = comm.allreduce(
        llvm_audit["basic_audit_passed"],
        op=MPI.LAND,
    )

    if rank == 0:
        gpu = cuda.get_current_device()

        manifest = {
            "assignment": "Hybrid MPI CUDA Cache Lab",
            "github_username": USERNAME,
            "generated_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "environment": {
                "platform": platform.platform(),
                "cpu_architecture": platform.machine(),
                "cache_line_bytes": get_cache_line_size(),
                "gpu": gpu.name.decode()
                if isinstance(gpu.name, bytes)
                else str(gpu.name),
                "mpi_ranks": comm.Get_size(),
            },
            "task3": {
                "steps": task3["steps"],
                "serialized_ms": task3["serialized_ms"],
                "overlapped_ms": task3["overlapped_ms"],
                "speedup": task3["speedup"],
                "maximum_error": task3["max_error"],
                "serialized_energy": task3["serialized_energy"],
                "overlapped_energy": task3["overlapped_energy"],
                "verification_passed": task3[
                    "verification_passed"
                ],
            },
            "task4": {
                "iterations_per_rank": CACHE_ITERATIONS,
                "strides": list(CACHE_STRIDES),
                "previous_observed_results_ms": {
                    "1": 508.585,
                    "8": 149.820,
                    "16": 200.115,
                },
                "previous_counter_check_passed": True,
                "llvm_audit": llvm_audit,
                "all_ranks_basic_llvm_audit_passed": bool(
                    all_llvm_pass
                ),
            },
            "verification": {
                "task3_parity_passed": bool(
                    task3["verification_passed"]
                ),
                "task4_basic_llvm_audit_passed": bool(
                    all_llvm_pass
                ),
                "full_assignment_verified": False,
                "note": (
                    "Task 1 and Task 2 passed earlier tests. "
                    "Task 4 timings are from the previous run. "
                    "Full independent reference validation and "
                    "LLVM loop-placement verification remain."
                ),
            },
        }

        with open(
            "submission_manifest.json", "w"
        ) as f:
            json.dump(manifest, f, indent=2)

        print("\nFINAL AUDIT RESULTS")
        print("Task 3 parity:", manifest[
            "verification"
        ]["task3_parity_passed"])
        print("Task 4 LLVM:", bool(all_llvm_pass))
        print("Cache line:", manifest[
            "environment"
        ]["cache_line_bytes"])
        print("Manifest: submission_manifest.json")
        print("FINAL AUDIT COMPLETED", flush=True)

finally:
    comm.Barrier()
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing task5_final.py


In [34]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python task5_final.py

In [35]:
import subprocess
from pathlib import Path

command = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python", "task5_final.py"
]

print("Script exists:", Path("task5_final.py").exists())

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    timeout=180,
)

print("EXIT CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-6000:])
print("\nSTDERR:\n", result.stderr[-6000:])
print(
    "\nMANIFEST EXISTS:",
    Path("submission_manifest.json").exists()
)

Script exists: True
EXIT CODE: 1

STDOUT:
 

STDERR:
 

MANIFEST EXISTS: False


In [36]:
import subprocess

tests = {
    "Python": ["python", "-c", "print('PYTHON OK')"],
    "MPI": [
        "mpirun", "--allow-run-as-root",
        "--oversubscribe", "-np", "4",
        "python", "-c",
        "from mpi4py import MPI; print('RANK', MPI.COMM_WORLD.Get_rank(), flush=True)"
    ],
    "Module import": [
        "python", "-c",
        "import hybrid_mpi_cuda_cache; print('IMPORT OK')"
    ],
}

for name, command in tests.items():
    print(f"\n=== {name} ===", flush=True)

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30
        )

        print("Exit code:", result.returncode)
        print("STDOUT:", result.stdout or "(empty)")
        print("STDERR:", result.stderr[-3000:] or "(empty)")

    except subprocess.TimeoutExpired:
        print("TIMEOUT")


=== Python ===
Exit code: 0
STDOUT: PYTHON OK

STDERR: (empty)

=== MPI ===
Exit code: 1
STDOUT: (empty)
STDERR: (empty)

=== Module import ===
Exit code: 0
STDOUT: IMPORT OK

STDERR: (empty)


In [37]:
!python -u task5_final.py

Traceback (most recent call last):
  File "/content/task5_final.py", line 60, in <module>
    ctx = init_hybrid_context(USERNAME)
  File "/content/hybrid_mpi_cuda_cache.py", line 19, in init_hybrid_context
    raise RuntimeError("Exactly 4 MPI ranks are required")
RuntimeError: Exactly 4 MPI ranks are required


In [38]:
import subprocess

print("MPI processes:")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,ppid,stat,cmd | grep -E 'mpirun|orted|prted|task5_final' | grep -v grep"],
    capture_output=True,
    text=True
)
print(result.stdout or "No MPI processes found")

print("\nMPI environment:")
result = subprocess.run(
    ["mpirun", "--version"],
    capture_output=True,
    text=True
)
print(result.stdout)

MPI processes:
   5157     565 Ssl  orted --hnp --set-sid --report-uri 86 --singleton-died-pipe 87 -mca state_novm_select 1 -mca ess hnp -mca pmix ^s1,s2,cray,isolated


MPI environment:
mpirun (Open MPI) 4.1.6

Report bugs to http://www.open-mpi.org/community/help/



In [39]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "plm", "isolated",
    "--mca", "btl", "self,vader,tcp",
    "-np", "4",
    "python", "-u", "-c",
    "from mpi4py import MPI; "
    "print('Rank', MPI.COMM_WORLD.Get_rank(), flush=True)"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=30
)

print("EXIT CODE:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr[-4000:])

EXIT CODE: 1
STDOUT:
 
STDERR:
 


In [40]:
import subprocess

subprocess.run(["pkill", "-f", "^orted "], check=False)
subprocess.run(["pkill", "-f", "^prted "], check=False)

print("Old MPI daemons cleanup completed")

Old MPI daemons cleanup completed


In [42]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python -u mpi_test.py

In [43]:
from google.colab import files
from pathlib import Path
import zipfile

names = [
    "hybrid_mpi_cuda_cache.py",
    "hybrid_mpi_cuda_cache_lab.ipynb",
    "task5_final.py",
    "task4_test.py",
]

with zipfile.ZipFile("hybrid_lab_backup.zip", "w") as z:
    for name in names:
        if Path(name).exists():
            z.write(name)
            print("Saved:", name)

files.download("hybrid_lab_backup.zip")

Saved: hybrid_mpi_cuda_cache.py
Saved: task5_final.py
Saved: task4_test.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!mpirun --version
!mpirun --allow-run-as-root --oversubscribe -np 4 hostname

mpirun (Open MPI) 4.1.6

Report bugs to http://www.open-mpi.org/community/help/
f26c34ef6ef9
f26c34ef6ef9
f26c34ef6ef9
f26c34ef6ef9


In [2]:
from pathlib import Path

for name in [
    "hybrid_mpi_cuda_cache.py",
    "task5_final.py",
    "task4_test.py",
]:
    print(name, "OK" if Path(name).exists() else "MISSING")

hybrid_mpi_cuda_cache.py OK
task5_final.py OK
task4_test.py OK


In [3]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python -u task5_final.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

In [4]:
import json

with open("submission_manifest.json") as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

{
  "assignment": "Hybrid MPI CUDA Cache Lab",
  "github_username": "230103369-bit",
  "generated_at_utc": "2026-11-20T04:02:18.147081+00:00",
  "environment": {
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "cpu_architecture": "x86_64",
    "cache_line_bytes": 64,
    "gpu": "Tesla T4",
    "mpi_ranks": 4
  },
  "task3": {
    "steps": 300,
    "serialized_ms": 1377.3828500000036,
    "overlapped_ms": 1700.7544409998445,
    "speedup": 0.8098657964933866,
    "maximum_error": 0.0,
    "serialized_energy": 65438.79229824849,
    "overlapped_energy": 65438.79229824849,
    "verification_passed": true
  },
  "task4": {
    "iterations_per_rank": 40000000,
    "strides": [
      1,
      8,
      16
    ],
    "previous_observed_results_ms": {
      "1": 508.585,
      "8": 149.82,
      "16": 200.115
    },
    "previous_counter_check_passed": true,
    "llvm_audit": {
      "i64_loads": 2,
      "i64_stores": 1,
      "memory_barriers": 2,
      "basic_audit_passed": true,

In [5]:
import numpy as np
from hybrid_mpi_cuda_cache import mpi_false_sharing_worker

dummy = np.zeros(64, dtype=np.uint64)
mpi_false_sharing_worker(dummy, 0, 10)

sig = mpi_false_sharing_worker.signatures[0]
llvm = mpi_false_sharing_worker.inspect_llvm(sig)

lines = llvm.splitlines()

for i, line in enumerate(lines):
    if "store i64" in line or "asm sideeffect" in line:
        print(f"\nLLVM line {i + 1}:")
        print("\n".join(lines[max(0, i - 4):i + 5]))


LLVM line 25:
  br label %B30

B30:                                              ; preds = %B30.lr.ph, %B30
  %lsr.iv = phi i64 [ %0, %B30.lr.ph ], [ %lsr.iv.next, %B30 ]
  tail call void asm sideeffect "", "~{memory}"() #2
  %sunkaddr = mul i64 %arg.slot, 8
  %sunkaddr1 = getelementptr i8, ptr %arg.shared.4, i64 %sunkaddr
  %.151 = load i64, ptr %sunkaddr1, align 8
  %.152 = add i64 %.151, 1

LLVM line 30:
  %sunkaddr = mul i64 %arg.slot, 8
  %sunkaddr1 = getelementptr i8, ptr %arg.shared.4, i64 %sunkaddr
  %.151 = load i64, ptr %sunkaddr1, align 8
  %.152 = add i64 %.151, 1
  store i64 %.152, ptr %sunkaddr1, align 8
  tail call void asm sideeffect "", "~{memory}"() #2
  %lsr.iv.next = add i64 %lsr.iv, -1
  %.103 = icmp samesign ugt i64 %lsr.iv.next, 1
  br i1 %.103, label %B30, label %B58

LLVM line 31:
  %sunkaddr1 = getelementptr i8, ptr %arg.shared.4, i64 %sunkaddr
  %.151 = load i64, ptr %sunkaddr1, align 8
  %.152 = add i64 %.151, 1
  store i64 %.152, ptr %sunkaddr1, align 8
  

In [6]:
import json

path = "submission_manifest.json"

with open(path) as f:
    manifest = json.load(f)

manifest["task4"]["llvm_audit"]["loop_store_placement_verified"] = True

with open(path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Task 4 LLVM loop placement: VERIFIED")
print("Manifest updated successfully")

Task 4 LLVM loop placement: VERIFIED
Manifest updated successfully


In [7]:
import json
import numpy as np

with open("submission_manifest.json") as f:
    manifest = json.load(f)

# Независимая проверка сохранённых результатов и статусов
assert manifest["verification"]["task3_parity_passed"]
assert manifest["verification"]["task4_basic_llvm_audit_passed"]
assert manifest["task4"]["llvm_audit"]["loop_store_placement_verified"]

print("Task 3 parity: PASSED")
print("Task 4 LLVM: PASSED")
print("Recorded benchmark:", manifest["task3"])
print("Independent CPU reference: NOT YET VERIFIED")

Task 3 parity: PASSED
Task 4 LLVM: PASSED
Recorded benchmark: {'steps': 300, 'serialized_ms': 1377.3828500000036, 'overlapped_ms': 1700.7544409998445, 'speedup': 0.8098657964933866, 'maximum_error': 0.0, 'serialized_energy': 65438.79229824849, 'overlapped_energy': 65438.79229824849, 'verification_passed': True}
Independent CPU reference: NOT YET VERIFIED


In [8]:
import inspect
import hybrid_mpi_cuda_cache as lab

for name, obj in inspect.getmembers(lab, inspect.isfunction):
    if any(word in name.lower() for word in
           ("stencil", "reference", "serial", "step", "initial")):
        print(f"\nFUNCTION: {name}")
        print("SIGNATURE:", inspect.signature(obj))
        try:
            print(inspect.getsource(obj)[:5000])
        except (OSError, TypeError):
            print("Source unavailable")


FUNCTION: benchmark_300_steps
SIGNATURE: (ctx)
def benchmark_300_steps(ctx):
    cart = ctx["cart_comm"]
    rank = ctx["rank"]

    serial = create_pipeline(ctx)
    overlap = create_pipeline(ctx)

    # Compile all CUDA kernels before benchmarking.
    warmup_serial = create_pipeline(ctx)
    warmup_overlap = create_pipeline(ctx)

    step_serialized(ctx, warmup_serial)
    step_overlapped(ctx, warmup_overlap)

    cuda.synchronize()
    cart.Barrier()

    def run_mode(pipeline, step_function):
        cart.Barrier()
        start = time.perf_counter()

        for _ in range(300):
            step_function(ctx, pipeline)

        cuda.synchronize()
        cart.Barrier()

        local_ms = (time.perf_counter() - start) * 1000

        return cart.allreduce(local_ms, op=MPI.MAX)

    serialized_ms = run_mode(serial, step_serialized)
    overlapped_ms = run_mode(overlap, step_overlapped)

    a = serial["u"].copy_to_host()
    b = overlap["u"].copy_to_host()

    local_error = floa

In [9]:
from pathlib import Path

code = Path("hybrid_mpi_cuda_cache.py").read_text().splitlines()

for i, line in enumerate(code):
    if line.startswith("def benchmark_300_steps") or line.startswith("def test_dual_stream"):
        print(f"\n{'=' * 20} {line} {'=' * 20}")
        print("\n".join(code[i:i+130]))


==================== def test_dual_stream(ctx): ====================
def test_dual_stream(ctx):
    serial = create_pipeline(ctx)
    overlap = create_pipeline(ctx)

    cart = ctx["cart_comm"]

    cart.Barrier()
    start = time.perf_counter()

    for _ in range(3):
        step_serialized(ctx, serial)

    cart.Barrier()
    serial_ms = (time.perf_counter() - start) * 1000

    cart.Barrier()
    start = time.perf_counter()

    for _ in range(3):
        step_overlapped(ctx, overlap)

    cart.Barrier()
    overlap_ms = (time.perf_counter() - start) * 1000

    a = serial["u"].copy_to_host()
    b = overlap["u"].copy_to_host()

    local_diff = float(
        np.max(np.abs(a[1:257, 1:257] - b[1:257, 1:257]))
    )

    max_diff = cart.allreduce(local_diff, op=MPI.MAX)

    assert max_diff < 1e-12, (
        f"Serialized/overlapped mismatch: {max_diff}"
    )

    max_serial_ms = cart.allreduce(serial_ms, op=MPI.MAX)
    max_overlap_ms = cart.allreduce(overlap_ms, op=MPI.MAX)

   

In [10]:
import inspect
import hybrid_mpi_cuda_cache as lab

print(inspect.getsource(lab.benchmark_300_steps)[:12000])

def benchmark_300_steps(ctx):
    cart = ctx["cart_comm"]
    rank = ctx["rank"]

    serial = create_pipeline(ctx)
    overlap = create_pipeline(ctx)

    # Compile all CUDA kernels before benchmarking.
    warmup_serial = create_pipeline(ctx)
    warmup_overlap = create_pipeline(ctx)

    step_serialized(ctx, warmup_serial)
    step_overlapped(ctx, warmup_overlap)

    cuda.synchronize()
    cart.Barrier()

    def run_mode(pipeline, step_function):
        cart.Barrier()
        start = time.perf_counter()

        for _ in range(300):
            step_function(ctx, pipeline)

        cuda.synchronize()
        cart.Barrier()

        local_ms = (time.perf_counter() - start) * 1000

        return cart.allreduce(local_ms, op=MPI.MAX)

    serialized_ms = run_mode(serial, step_serialized)
    overlapped_ms = run_mode(overlap, step_overlapped)

    a = serial["u"].copy_to_host()
    b = overlap["u"].copy_to_host()

    local_error = float(
        np.max(
            np.abs(
         

In [11]:
import json

with open("submission_manifest.json") as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

{
  "assignment": "Hybrid MPI CUDA Cache Lab",
  "github_username": "230103369-bit",
  "generated_at_utc": "2026-11-20T04:02:18.147081+00:00",
  "environment": {
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "cpu_architecture": "x86_64",
    "cache_line_bytes": 64,
    "gpu": "Tesla T4",
    "mpi_ranks": 4
  },
  "task3": {
    "steps": 300,
    "serialized_ms": 1377.3828500000036,
    "overlapped_ms": 1700.7544409998445,
    "speedup": 0.8098657964933866,
    "maximum_error": 0.0,
    "serialized_energy": 65438.79229824849,
    "overlapped_energy": 65438.79229824849,
    "verification_passed": true
  },
  "task4": {
    "iterations_per_rank": 40000000,
    "strides": [
      1,
      8,
      16
    ],
    "previous_observed_results_ms": {
      "1": 508.585,
      "8": 149.82,
      "16": 200.115
    },
    "previous_counter_check_passed": true,
    "llvm_audit": {
      "i64_loads": 2,
      "i64_stores": 1,
      "memory_barriers": 2,
      "basic_audit_passed": true,

### Analytical & Forensic Engineering Responses

**Question 1 — GPU Coalescing vs. CPU Cache-Line Granularity**

GPU memory coalescing and CPU cache-line coherency optimize different aspects of memory access. On the NVIDIA Tesla T4, contiguous memory accesses by threads in a warp can be combined into efficient memory transactions. GPU L2 memory operates with 32-byte sectors, while a 128-byte region corresponds to four such sectors. Accessing a strided column in a row-major matrix can generate inefficient memory transactions, so packing boundary columns into contiguous buffers improves memory-transfer efficiency.

On the CPU, the measured cache-line size was 64 bytes, sufficient to hold eight 64-bit counters. When separate MPI ranks modify different counters within the same cache line, false sharing can occur even though the counters are logically independent.

**Question 2 — Shared-Memory Tile Synchronization & PTX Audit**

The CUDA stencil implementation uses a 16 × 16 computational tile with an 18 × 18 shared-memory region to include the surrounding halo. Shared-memory tiling reduces repeated global-memory accesses by allowing neighboring GPU threads to reuse loaded data.

Synchronization with `cuda.syncthreads()` is necessary after loading the tile and before reading values that may have been written by other threads. Without this barrier, threads could read incomplete or stale shared-memory data.

The PTX audit should confirm shared-memory load and store instructions and a block synchronization instruction such as `bar.sync`. The current manifest does not contain the PTX instruction counts, so those counts cannot be reported as measured results here.

**Question 3 — Dual-Stream Compute-Communication Overlap on Shared GPU Silicon**

The benchmark executed 300 stencil steps using four MPI ranks on an NVIDIA Tesla T4 GPU.

- Serialized execution: 1377.383 ms
- Overlapped execution: 1700.754 ms
- Speedup: 0.8099×
- Maximum numerical error: 0.0

The overlapped implementation was approximately 23.5% slower than the serialized implementation. This demonstrates that asynchronous execution does not automatically improve performance.

All four MPI ranks share the same physical GPU, so kernel scheduling, memory bandwidth, and transfer resources may become bottlenecks. Additional stream scheduling and synchronization overhead can also outweigh the benefits of overlapping computation with communication.

The zero maximum error confirms numerical equivalence between the two execution modes.

**Question 4 — MPI-3 Shared-Memory MESI RFO Ping-Pong**

The MPI-3 shared-memory benchmark evaluated four ranks incrementing independent 64-bit counters, each performing 40,000,000 increments.

| Stride | Execution time |
|---|---:|
| 1 | 508.585 ms |
| 8 | 149.820 ms |
| 16 | 200.115 ms |

With stride 1, adjacent counters occupy the same 64-byte cache line. When different CPU cores repeatedly write to this line, the MESI coherency protocol requires ownership transfers and invalidations through Read-For-Ownership operations.

This produces cache-line ping-pong despite the absence of logical counter sharing.

Increasing the stride to 8 places the counters on separate cache lines and reduced execution time by approximately 3.39× compared with stride 1.

The LLVM audit also identified 64-bit loads, 64-bit stores, and memory barriers, with an additional inspection confirming the store occurs inside the loop rather than only after loop completion.

**Question 5 — 128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs**

Stride 8 separates consecutive 64-bit counters by 64 bytes, matching the measured CPU cache-line size. Stride 16 separates them by 128 bytes.

Although both layouts avoid placing adjacent counters in the same 64-byte line, the observed times differed:

- Stride 8: 149.820 ms
- Stride 16: 200.115 ms

Stride 16 was approximately 33.6% slower than stride 8 in this run.

A possible explanation involves adjacent-line prefetching, cache-set placement, scheduling variability, and memory-system effects. Some processors may fetch or track neighboring 64-byte lines together, which can influence performance even when the counters do not directly share a cache line.

However, this experiment alone does not establish that a 128-byte spatial prefetcher caused the difference. Additional repeated measurements and hardware performance counters would be necessary to isolate the mechanism.

Padding also increases memory consumption, so larger strides are not automatically better.

**Question 6 — CFL Stability & Distributed Energy Dissipation**

For an explicit two-dimensional diffusion stencil with equal spatial spacing, the standard stability condition is:

\[
\frac{\kappa_{\max}\Delta t}{\Delta x^2}\leq\frac14
\]

This condition limits the time step relative to the maximum diffusion coefficient and grid spacing. Exceeding the stability limit can cause numerical errors to grow rather than decay.

The distributed MPI implementation must exchange halo values consistently so that stencil updates at subdomain boundaries use the correct neighboring data.

In the 300-step benchmark, the recorded total energy was identical for both execution modes:

- Serialized energy: 65438.79229824849
- Overlapped energy: 65438.79229824849

The maximum difference between the resulting fields was 0.0, demonstrating consistency between serialized and overlapped execution.

These results do not independently prove that the CFL condition was satisfied or that energy decreased monotonically at every step. Confirming those properties requires the actual numerical parameters and an energy history or independent reference calculation.

**Conclusion**

The experiments demonstrate two important performance principles. First, overlapping MPI communication and CUDA computation can reduce performance when multiple ranks compete for the same GPU. Second, CPU false sharing can significantly increase execution time even when different ranks modify independent variables.

The numerical parity and LLVM checks passed, while the full independent reference verification remains incomplete.